# 🎬 CreatorOps Video & Reaction Agent Notebook

This notebook provides a complete automated pipeline for downloading YouTube videos, compositing split-screen reaction shorts, floating PiP overlays, and background audio streams.

Use media you own or have permission to edit.

In [ ]:
from pathlib import Path
import sys
import tempfile

# Locate CreatorAgent root directory regardless of execution folder
current_dir = Path.cwd().resolve()
creator_agent_dir = None

for p in [current_dir, *current_dir.parents]:
    if p.name == 'CreatorAgent':
        creator_agent_dir = p
        break
    elif (p / 'CreatorAgent').exists():
        creator_agent_dir = p / 'CreatorAgent'
        break

if not creator_agent_dir:
    creator_agent_dir = current_dir

if str(creator_agent_dir) not in sys.path:
    sys.path.insert(0, str(creator_agent_dir))

from Backend.video_processor import (
    build_video,
    build_video_advanced,
    download_youtube,
    save_upload,
    estimate_processing_time
)

print(f'✅ CreatorOps Agent initialized at: {creator_agent_dir}')

In [ ]:
# Input Configuration
# Set YOUTUBE_URL to download directly from YouTube, or set LOCAL_SOURCE to a local file path.
YOUTUBE_URL = 'https://www.youtube.com/watch?v=jNQXAC9IVRw'
LOCAL_SOURCE = None  # Example: creator_agent_dir / 'media' / 'source.mp4'

# Optional Secondary Video (for Split-Screen / Reaction / PiP)
YOUTUBE_URL_2 = ''  # Second YouTube URL
LOCAL_SOURCE_2 = None

# Browser Sign-in Option for YouTube (Chrome, Edge, Firefox, Brave, or None)
BROWSER = 'Chrome'

# Editing & Canvas Settings
START_SECONDS = 0.0
END_SECONDS = 15.0  # Clip end duration in seconds
RATIO = '9:16'  # Aspect ratios: '9:16', '16:9', '1:1', or '4:5'
LAYOUT = 'single'  # Layout modes: 'single', 'top_landscape', 'split_v', 'split_h', 'pip'
BACKGROUND_MUSIC = None  # Optional audio path: creator_agent_dir / 'media' / 'music.mp3'
MUSIC_VOLUME = 0.18

In [ ]:
# Execute Video Processing Pipeline
work_dir = creator_agent_dir / 'media' / 'temp_workspace'
work_dir.mkdir(parents=True, exist_ok=True)

# Process Video 1
if YOUTUBE_URL.strip():
    print(f'📥 Downloading Video 1 from YouTube: {YOUTUBE_URL}...')
    source1 = download_youtube(
        url=YOUTUBE_URL.strip(),
        output_dir=work_dir,
        browser=BROWSER,
        start_seconds=START_SECONDS,
        end_seconds=END_SECONDS,
        filename_prefix='v1_source'
    )
elif LOCAL_SOURCE and Path(LOCAL_SOURCE).exists():
    source1 = Path(LOCAL_SOURCE)
else:
    raise ValueError('Please specify a valid YOUTUBE_URL or LOCAL_SOURCE path for Video 1.')

# Process Video 2 (Optional)
source2 = None
if YOUTUBE_URL_2.strip():
    print(f'📥 Downloading Video 2 from YouTube: {YOUTUBE_URL_2}...')
    source2 = download_youtube(
        url=YOUTUBE_URL_2.strip(),
        output_dir=work_dir,
        browser=BROWSER,
        start_seconds=0.0,
        end_seconds=END_SECONDS,
        filename_prefix='v2_source'
    )
elif LOCAL_SOURCE_2 and Path(LOCAL_SOURCE_2).exists():
    source2 = Path(LOCAL_SOURCE_2)

music = Path(BACKGROUND_MUSIC) if (BACKGROUND_MUSIC and Path(BACKGROUND_MUSIC).exists()) else None
output = creator_agent_dir / 'media' / 'creatorops-notebook-final.mp4'
output.parent.mkdir(parents=True, exist_ok=True)

print('🚀 Building final video render...')
build_video_advanced(
    source1=source1,
    source2=source2,
    music=music,
    output=output,
    layout=LAYOUT,
    ratio=RATIO,
    v1_start=START_SECONDS,
    v1_end=END_SECONDS,
    music_volume=MUSIC_VOLUME
)

print(f'✨ Export complete! Final video saved to: {output}')

In [ ]:
try:
    from IPython.display import Video, display
    if 'output' in locals() and output.exists():
        display(Video(str(output), embed=True))
    else:
        print('No output video file generated yet.')
except ImportError:
    if 'output' in locals() and output.exists():
        print(f'Rendered video ready at: {output}')
    else:
        print('No output video file generated yet.')